# Basin selection in EML expression trees — walkthrough

This notebook reproduces the key results of the v2.4 warm-start note from the released data, then runs one live warm-start training so you can watch a valid snap happen.

Setup: `pip install -e .` from the repo root (or just run the notebook from `examples/` with the repo root on `sys.path`).

In [ ]:
import sys, csv
from pathlib import Path
from collections import defaultdict
ROOT = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
sys.path.insert(0, str(ROOT))
import torch
from eml_layer_v2 import EMLTree, train_eml

## 1. The released data: valid-snap rates per (function, depth, init mode)

The canonical v2.4 dataset is 120 runs: 20 seeds × {exp d=3, ln d=5} × {blind, warm, curriculum}. `valid_snap` = every selector at a simplex vertex **and** post-snap MAE < 0.01.

In [ ]:
cells = defaultdict(lambda: [0, 0, set()])
with open(ROOT / 'results' / 'basin_warmstart_v2.4_postfix.csv', newline='') as f:
    for r in csv.DictReader(f):
        key = (r['function'], int(r['depth']), r['init_mode'])
        cells[key][1] += 1
        if r['valid_snap'] == '1':
            cells[key][0] += 1
            cells[key][2].add(r['symbolic_form'])
for (func, d, mode), (v, n, forms) in sorted(cells.items()):
    print(f'{func:<5} d={d} {mode:<11} {v:>3}/{n:<3} {sorted(forms)}')

Blind training recovers the correct form only 25–35% of the time; warm-start (`initialize_to_target`) and curriculum recover 20/20. Commitment was never the problem — basin selection during phase 1 is.

## 2. One live run: warm-started ln at depth 5 (over-representational)

ln(x) = `eml(1, eml(eml(1,x), 1))` needs depth 4 in this balanced-tree notation. At depth 5 the blind valid rate drops; the warm start plants the 3-gate core at the top levels and biases everything else to constants. (~2–3 min on CPU.)

In [ ]:
torch.manual_seed(0)
x = torch.linspace(0.1, 10.0, 128)
y = torch.log(x)
tree = EMLTree(depth=5)
tree.initialize_to_target('ln', noise=0.4)
print('pretrain form:', tree.symbolic_form())
m = train_eml(tree, x, y, epochs=2000, lr=0.0005)
print('valid_snap:', m['valid_snap'])
print('final form:', tree.symbolic_form())
print('post-snap MAE:', m['post_snap_loss'])

## 3. Curriculum: grow a trained shallow solution into a deeper tree

`grow_from_shallow` embeds a trained depth-4 ln solution **top-aligned** into the depth-5 tree (v2.5): the extra capacity dangles at the bottom as constants, so the embedded value is preserved exactly — no identity-forwarding gate needed.

In [ ]:
torch.manual_seed(1)
shallow = EMLTree(depth=4)
shallow.initialize_to_target('ln', noise=0.3)
m_sh = train_eml(shallow, x, y, epochs=1000, lr=0.001)
shallow.snap_all()
print('shallow valid:', m_sh['valid_snap'], shallow.symbolic_form())

deep = EMLTree(depth=5)
deep.grow_from_shallow(shallow, noise=0.3)
print('deep pretrain form:', deep.symbolic_form())
m = train_eml(deep, x, y, epochs=2000, lr=0.0005)
print('deep valid:', m['valid_snap'], deep.symbolic_form())